# Part 2 — TF-IDF + ChiSqSelector via the Spark ML DataFrame pipeline

Builds the canonical Assignment 2 feature pipeline and writes the 2000 ChiSqSelector-picked vocabulary terms to `outputs/output_ds.txt` (one alphabetical, space-separated line, mirroring the merged-dictionary line at the bottom of Assignment 1's `output.txt` so set comparison is direct).

All pipeline logic lives in [pipeline.py](pipeline.py); this notebook is a thin driver that imports and calls it, so script and notebook stay in lock-step.

## Pipeline overview

1. `RegexTokenizer` — same delimiter pattern as Part 1 (`REGEX_TOKENIZER_PATTERN` from `common.text_utils`), `gaps=True`, `toLowercase=True`, `minTokenLength=2`.
2. `StopWordsRemover` — seeded with the 596 entries in `common/stopwords.txt`.
3. `CountVectorizer` — raw term-frequency counts (not `HashingTF`; we need the vocabulary back for `output_ds.txt`).
4. `IDF` — multiplies TF by inverse document frequency.
5. `StringIndexer` — numeric label column, required by `ChiSqSelector`.
6. `ChiSqSelector(numTopFeatures=2000)` — picks the 2000 highest-chi² vocabulary indices.

Selected indices are looked up in `CountVectorizerModel.vocabulary` and written sorted to `output_ds.txt`.

The factory `build_feature_pipeline(num_top_features)` is the contract Part 3 plugs into for the grid search.

## Imports

Make `src/` importable so both `common/` and `part2_pipeline/` resolve.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
SRC_DIR = (NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'part2_pipeline' else NOTEBOOK_DIR / 'src')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from part2_pipeline import pipeline

## Local run against the dev sample

Adjust paths if your checkout layout differs. The dev sample lives in Assignment 1's assets folder.

In [ ]:
ASSIGNMENT_2 = SRC_DIR.parent
INPUT_PATH = str(ASSIGNMENT_2.parent / 'Assignment 1' / 'src' / 'Assignment_1_Assets' / 'reviews_devset.json')
STOPWORDS_PATH = str(ASSIGNMENT_2 / 'src' / 'common' / 'stopwords.txt')
OUTPUT_PATH = str(ASSIGNMENT_2 / 'outputs' / 'output_ds.txt')

print('input    :', INPUT_PATH)
print('stopwords:', STOPWORDS_PATH)
print('output   :', OUTPUT_PATH)

In [ ]:
pipeline.run(
    input_path=INPUT_PATH,
    stopwords_path=STOPWORDS_PATH,
    output_path=OUTPUT_PATH,
    mode='local',
    num_top_features=2000,
)

## Quick sanity checks

- The file should be a single line.
- That line should contain exactly 2000 unique alphabetical terms.
- No delimiter character should survive inside any term.

In [ ]:
from common.text_utils import DELIMITER_CHARS

text = Path(OUTPUT_PATH).read_text(encoding='utf-8')
lines = text.splitlines()
terms = lines[0].split(' ') if lines else []

print(f'lines        : {len(lines)}')
print(f'terms        : {len(terms)}')
print(f'unique terms : {len(set(terms))}')
print(f'sorted       : {terms == sorted(terms)}')

bad = [t for t in terms if any(c in t for c in DELIMITER_CHARS)]
print(f'tokens with surviving delimiter chars: {len(bad)}')
if bad:
    print(f'  examples: {bad[:5]}')

## Comparison vs Assignment 1

Set overlap between our 2000 ChiSqSelector picks and the 1418 terms on the last (merged-dictionary) line of `Assignment_1/output.txt`. The spec explicitly notes: results from Part 2 will *not* be identical to Part 1 / Assignment 1 — the two select features differently (top-K per category vs. global TF-IDF + chi²). The overlap and divergence shape the report's comparison paragraph; they are not pass/fail signals.

In [ ]:
REF_PATH = ASSIGNMENT_2 / 'Assignment_1' / 'output.txt'
ref_lines = REF_PATH.read_text(encoding='utf-8').splitlines()
ref_merged = set(ref_lines[-1].split())  # the alphabetical merged-dictionary line
ours = set(terms)

shared = ours & ref_merged
only_ours = ours - ref_merged
only_ref = ref_merged - ours

print(f'ours    : {len(ours)} terms')
print(f'ref     : {len(ref_merged)} terms')
print(f'overlap : {len(shared)} ({100 * len(shared) / len(ref_merged):.1f}% of ref)')
print(f'only ours (sample): {sorted(only_ours)[:10]}')
print(f'only ref  (sample): {sorted(only_ref)[:10]}')

## Cluster run

Don't run on the cluster from inside this notebook — use `spark-submit` from a terminal so the job goes to YARN cluster mode and reads from HDFS:

```bash
bash scripts/build_common_zip.sh
spark-submit --master yarn --deploy-mode cluster \
  --py-files common.zip \
  --files src/common/stopwords.txt \
  src/part2_pipeline/pipeline.py \
  --mode cluster \
  --input hdfs:///dic_shared/amazon-reviews/full/reviews_devset.json \
  --stopwords stopwords.txt \
  --output output_ds.txt
```